# Demo inference server (Colab)

Serves:
- **Phi-3-mini-4k-instruct** with 2 LoRA adapters (switchable per request)
- **GPT-2** fine-tuned model

Exposes a FastAPI HTTP server through an ngrok tunnel so your existing FastAPI backend can call it as a drop-in remote model provider.

**Runtime -> Change runtime type -> GPU (T4)** before running.

## 1. Install dependencies

In [1]:
!pip install -q -U transformers accelerate peft fastapi uvicorn pyngrok nest_asyncio  unsloth

## 2. (Optional) Mount Google Drive for adapter / GPT-2 weights

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Config — edit these paths

- `ADAPTER1_PATH` / `ADAPTER2_PATH`: folders with `adapter_config.json` + adapter weights (PEFT LoRA output).
- `GPT2_MODEL_PATH`: either a local/Drive folder with your fine-tuned GPT-2, or a Hugging Face hub id (e.g. `"gpt2"` for the vanilla model if you don't have a fine-tuned one yet).

In [2]:
BASE_MODEL = "microsoft/Phi-3-mini-4k-instruct"
ADAPTER1_PATH = "/kaggle/input/models/ahmedmohamedfarag111/test/vllm/default/1/posts_adapter"
ADAPTER2_PATH = "/kaggle/input/models/ahmedmohamedfarag111/test2/vllm/default/1/phi-lora-model-blogs"

## 4. Load models (once)

In [3]:
import torch
from unsloth import FastLanguageModel
from peft import PeftModel

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# Load Phi-3 base model using Unsloth — force single GPU, no auto device_map
phi_model, phi_tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=16384,
    dtype=torch.bfloat16,
    load_in_4bit=False,
    device_map={"": 0},   # <-- pin everything to cuda:0, no auto split
)

phi_model = PeftModel.from_pretrained(
    phi_model,
    ADAPTER1_PATH,
    adapter_name="adapter1",
)

phi_model.load_adapter(
    ADAPTER2_PATH,
    adapter_name="adapter2",
)

phi_model.set_adapter("adapter1")

FastLanguageModel.for_inference(phi_model)
phi_model.eval()

PHI_ADAPTERS = ["adapter1", "adapter2"]

print("Models loaded.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!
Device: cuda:0
==((====))==  Unsloth 2026.7.2: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


[unsloth_zoo.log|WARNING]Device does not support bfloat16. Will change to float16.
[unsloth_zoo.log|WARNING]Unsloth: unsloth/Phi-3-mini-4k-instruct can only handle sequence lengths of at most 4096.
But with kaiokendev's RoPE scaling of 4.0, it can be magically be extended to 16384!
[unsloth_zoo.log|WARNING]Unsloth: Could not apply RoPE scaling 'linear' from config (NotImplementedError: Cannot copy out of meta tensor; no data!); falling back to unscaled RoPE. Long-context generation may degrade.


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Could not apply RoPE scaling 'linear' from config (TypeError: unsupported operand type(s) for ** or pow(): 'NoneType' and 'Tensor'); falling back to unscaled RoPE. Long-context generation may degrade.


Models loaded.


## 5. Generation helpers

One global lock serializes all generation calls. This is the simplest way to avoid two requests colliding on `set_adapter()` or overlapping GPU work on a single Colab GPU — fine for a demo.

In [4]:
import threading

generation_lock = threading.Lock()

def generate_phi(prompt: str, adapter: str, max_new_tokens: int, temperature: float, top_p: float) -> str:
    """Single-turn: wraps a plain prompt as one user message."""
    return generate_phi_chat(
        [{"role": "user", "content": prompt}],
        adapter, max_new_tokens, temperature, top_p,
    )

def generate_phi_chat(messages: list, adapter: str, max_new_tokens: int, temperature: float, top_p: float) -> str:
    """Takes a full messages list (system + user, etc.) and preserves all of it."""
    if adapter not in PHI_ADAPTERS:
        raise ValueError(f"Unknown adapter '{adapter}'. Available: {PHI_ADAPTERS}")

    text = phi_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = phi_tokenizer(text, return_tensors="pt").to("cuda:0")

    with generation_lock:
        phi_model.set_adapter(adapter)
        with torch.no_grad():
            output_ids = phi_model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_p=top_p,
                do_sample=True,
                pad_token_id=phi_tokenizer.eos_token_id,
            )

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return phi_tokenizer.decode(new_tokens, skip_special_tokens=True)

## 6. FastAPI app

In [5]:
import time
import uuid
from typing import List, Literal, Optional

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

app = FastAPI(title="Demo Inference Server")


@app.get("/health")
def health():
    return {"status": "healthy"}


@app.get("/models")
def list_models():
    return {
        "models": [
            {"name": "phi3", "adapters": PHI_ADAPTERS},
            {"name": "gpt2"},
        ]
    }


class GenerateRequest(BaseModel):
    model: Literal["phi3", "gpt2"]
    adapter: Optional[str] = None
    prompt: str
    max_new_tokens: int = 4000
    temperature: float = 0.7
    top_p: float = 0.9


class GenerateResponse(BaseModel):
    generated_text: str
    model: str
    adapter: Optional[str] = None


@app.post("/generate", response_model=GenerateResponse)
def generate(req: GenerateRequest):
    if req.model == "phi3":
        adapter = req.adapter or PHI_ADAPTERS[0]
        try:
            text = generate_phi(req.prompt, adapter, req.max_new_tokens, req.temperature, req.top_p)
        except ValueError as exc:
            raise HTTPException(status_code=400, detail=str(exc))
        return GenerateResponse(generated_text=text, model="phi3", adapter=adapter)

    if req.model == "gpt2":
        text = generate_gpt2(req.prompt, req.max_new_tokens)
        return GenerateResponse(generated_text=text, model="gpt2", adapter=None)

    raise HTTPException(status_code=400, detail=f"Unknown model '{req.model}'")


# --- OpenAI-compatible endpoint ---
# model field convention: "phi3", "phi3:adapter1", "phi3:adapter2", or "gpt2"

class ChatMessage(BaseModel):
    role: str
    content: str


class ChatCompletionRequest(BaseModel):
    model: str
    messages: List[ChatMessage]
    max_tokens: int = 4000
    temperature: float = 0.7
    top_p: float = 0.9


@app.post("/v1/chat/completions")
def chat_completions(req: ChatCompletionRequest):
    # Preserve the FULL message list (system + user), not just the last user turn.
    msg_dicts = [{"role": m.role, "content": m.content} for m in req.messages]

    if req.model.startswith("phi3"):
        adapter = req.model.split(":", 1)[1] if ":" in req.model else PHI_ADAPTERS[0]
        try:
            text = generate_phi_chat(msg_dicts, adapter, req.max_tokens, req.temperature, req.top_p)
        except ValueError as exc:
            raise HTTPException(status_code=400, detail=str(exc))
    elif req.model == "gpt2":
        last_user_msg = next((m.content for m in reversed(req.messages) if m.role == "user"), "")
        text = generate_gpt2(last_user_msg, req.max_tokens)
    else:
        raise HTTPException(status_code=400, detail=f"Unknown model '{req.model}'")

    return {
        "id": f"chatcmpl-{uuid.uuid4().hex[:24]}",
        "object": "chat.completion",
        "created": int(time.time()),
        "model": req.model,
        "choices": [
            {
                "index": 0,
                "message": {"role": "assistant", "content": text},
                "finish_reason": "stop",
            }
        ],
    }

## 7. Start the server + ngrok tunnel

You need a free ngrok account/authtoken: https://dashboard.ngrok.com/get-started/your-authtoken

This will prompt you to paste it in (hidden input, not saved to the notebook).

In [6]:
from getpass import getpass
from pyngrok import ngrok

ngrok_token = getpass("Paste your ngrok authtoken: ")
ngrok.set_auth_token(ngrok_token)

Paste your ngrok authtoken:  ········


In [7]:
import threading
import nest_asyncio
import uvicorn

nest_asyncio.apply()

PORT = 8000

def _run():
    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="info")

server_thread = threading.Thread(target=_run, daemon=True)
server_thread.start()

public_url = ngrok.connect(PORT, "http")
print("Public endpoint:", public_url)

INFO:     Started server process [1105]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Public endpoint: NgrokTunnel: "https://qualifier-agreement-ahead.ngrok-free.dev" -> "http://localhost:8000"


Both `max_new_tokens` (=4000) and `max_length`(=16384) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


INFO:     45.241.21.160:0 - "POST /v1/chat/completions HTTP/1.1" 200 OK


Both `max_new_tokens` (=4000) and `max_length`(=16384) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.

...

INFO:     45.241.21.160:0 - "POST /v1/chat/completions HTTP/1.1" 200 OK


Both `max_new_tokens` (=4000) and `max_length`(=16384) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INFO:     45.241.21.160:0 - "POST /v1/chat/completions HTTP/1.1" 200 OK


Both `max_new_tokens` (=4000) and `max_length`(=16384) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INFO:     45.241.21.160:0 - "POST /v1/chat/completions HTTP/1.1" 200 OK


In [ ]:
test_messages = [
    {"role": "system", "content": "You are a helpful assistant. Respond only in JSON."},
    {"role": "user", "content": "Say hi."},
]
print(phi_tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True))

## 8. Example requests

Replace `PUBLIC_URL` below with the URL printed above (e.g. `https://xxxx.ngrok-free.app`).

**curl:**
```bash
curl -X GET https://https://qualifier-agreement-ahead.ngrok-free.dev/health

curl -X GET https://https://qualifier-agreement-ahead.ngrok-free.dev/models

curl -X POST https://https://qualifier-agreement-ahead.ngrok-free.dev/generate \
  -H "Content-Type: application/json" \
  -d '{
    "model": "phi3",
    "adapter": "adapter1",
    "prompt": "Write a LinkedIn post about AI.",
    "max_new_tokens": 4000,
    "temperature": 0.7,
    "top_p": 0.9
  }'

curl -X POST https://https://qualifier-agreement-ahead.ngrok-free.dev/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{
    "model": "phi3:adapter1",
    "messages": [{"role": "user", "content": "Write a LinkedIn post about AI."}]
  }'
```

In [ ]:
import requests

# Your ngrok URL
PUBLIC_URL = "https://qualifier-agreement-ahead.ngrok-free.dev"

# Test the server
print(requests.get(f"{PUBLIC_URL}/health").json())
print(requests.get(f"{PUBLIC_URL}/models").json())

# Generate text
resp = requests.post(
    f"{PUBLIC_URL}/generate",
    json={
        "model": "phi3",
        "adapter": "adapter1",
        "prompt": "Write a LinkedIn post about AI.",
        "max_new_tokens": 200,
        "temperature": 0.7,
        "top_p": 0.9,
    },
    timeout=300,
)

print(resp.status_code)
print(resp.json())

## Pointing your FastAPI backend at this server

In your backend's settings/config, set the base URL to the `PUBLIC_URL` printed above (as an env var, not hardcoded, since it changes on every Colab restart). Then either:

- Call `/generate` directly with `model`/`adapter`/`prompt`, or
- If your agent graph's LLM client is OpenAI-compatible (e.g. `langchain_openai.ChatOpenAI` or the `openai` SDK), just point its `base_url` at `PUBLIC_URL` and set `model` to `"phi3"`, `"phi3:adapter2"`, or `"gpt2"` — no other code changes needed.

**Reminder:** this notebook needs to keep running for the demo to work — closing the Colab tab or letting it idle-disconnect will kill the tunnel.